In [1]:
!pip install -q google-generativeai pypdf reportlab numpy chromadb gradio pysqlite3-binary

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [2]:
# Fix Colab's sqlite3 so ChromaDB is happy (must run before importing chromadb)
try:
    __import__("pysqlite3")
    import sys
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
    print("Patched sqlite3 with pysqlite3.")
except Exception as e:
    print("pysqlite3 not needed or unavailable:", e)


Patched sqlite3 with pysqlite3.


In [3]:
import google.generativeai as genai

API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("Paste your Gemini API key: ")
genai.configure(api_key=API_KEY)

EMBED_MODEL = "models/gemini-embedding-001"
CHAT_MODEL  = "gemini-2.5-flash"
print("Configured.")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Paste your Gemini API key: ··········
Configured.


In [4]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas

DOCS_DIR = "clinic_docs"
os.makedirs(DOCS_DIR, exist_ok=True)

PATIENTS = [
    {"name":"John A. Carter","mrn":"MRN-100245","dob":"1958-03-12","age":68,
     "note":("Nephrology Progress Note. CKD Stage 5 on in-center hemodialysis 3x/week (Mon/Wed/Fri). "
             "Primary etiology: diabetic nephropathy. Dialysis access: left upper-arm AV fistula, functioning "
             "well, no signs of stenosis. Interdialytic weight gain averaging 2.4 kg. Reports mild fatigue and "
             "leg cramps during the last hour of sessions. Pre-dialysis blood pressure 152/88. Kt/V 1.5."),
     "labs":[("Hemoglobin","9.8 g/dL","Low"),("Potassium","5.6 mEq/L","High"),("Phosphorus","6.1 mg/dL","High"),
             ("Calcium","8.9 mg/dL","Normal"),("Albumin","3.6 g/dL","Low-normal"),("Kt/V","1.5","Adequate")],
     "meds":["Epoetin alfa 4000 units IV with dialysis","Sevelamer 800 mg PO TID with meals",
             "Calcitriol 0.25 mcg PO daily","Lisinopril 10 mg PO daily","Insulin glargine 22 units SC nightly"]},
    {"name":"Maria L. Gonzalez","mrn":"MRN-100871","dob":"1971-09-30","age":54,
     "note":("Outpatient Nephrology Visit. CKD Stage 4 (eGFR 22), not yet on dialysis. Etiology: hypertensive "
             "nephrosclerosis. Pre-emptive transplant evaluation in progress. Counseled on AV fistula vs "
             "peritoneal dialysis if function declines. Euvolemic. BP 138/84. Good adherence."),
     "labs":[("eGFR","22 mL/min/1.73m2","Low"),("Creatinine","2.8 mg/dL","High"),("Potassium","4.9 mEq/L","Normal"),
             ("Bicarbonate","20 mEq/L","Low"),("Hemoglobin","11.2 g/dL","Low-normal"),("Urine ACR","640 mg/g","High")],
     "meds":["Losartan 100 mg PO daily","Amlodipine 5 mg PO daily","Sodium bicarbonate 650 mg PO BID",
             "Furosemide 20 mg PO daily as needed"]},
    {"name":"Robert T. Nguyen","mrn":"MRN-101533","dob":"1949-06-05","age":76,
     "note":("Home Peritoneal Dialysis Follow-up. ESRD on automated peritoneal dialysis (APD) nightly. "
             "Etiology: polycystic kidney disease. PD catheter exit site clean. One episode of cloudy effluent "
             "two weeks ago treated for peritonitis with intraperitoneal antibiotics; culture grew "
             "coagulase-negative Staphylococcus, now resolved. Residual renal function preserved. BP 128/76."),
     "labs":[("Hemoglobin","10.4 g/dL","Low"),("Potassium","4.2 mEq/L","Normal"),("Phosphorus","4.8 mg/dL","Slightly high"),
             ("Albumin","3.9 g/dL","Normal"),("PD Kt/V (weekly)","1.8","Adequate"),("CRP","3.1 mg/L","Normal")],
     "meds":["Darbepoetin alfa 40 mcg SC every 2 weeks","Lanthanum carbonate 500 mg PO TID",
             "Cinacalcet 30 mg PO daily","Aspirin 81 mg PO daily"]},
]

def make_pdf(p):
    path = os.path.join(DOCS_DIR, p["mrn"].replace("-","_")+".pdf")
    c = canvas.Canvas(path, pagesize=letter); width, height = letter; y = height - inch
    def line(txt, dy=16, font="Helvetica", size=11):
        nonlocal y; c.setFont(font, size); c.drawString(inch, y, txt); y -= dy
    line("RIVER VALLEY NEPHROLOGY ASSOCIATES  -  EXTERNAL CLINIC RECORD",22,"Helvetica-Bold",12)
    line(f"Patient: {p['name']}    MRN: {p['mrn']}",16,"Helvetica-Bold",11)
    line(f"DOB: {p['dob']}    Age: {p['age']}",20)
    line("CLINICAL NOTE",18,"Helvetica-Bold",11)
    words, cur = p["note"].split(), ""
    for w in words:
        if len(cur)+len(w)+1 > 95: line(cur); cur = w
        else: cur = (cur+" "+w).strip()
    if cur: line(cur)
    y -= 6
    line("LABORATORY RESULTS",18,"Helvetica-Bold",11)
    for n,v,fl in p["labs"]: line(f"  - {n}: {v}  ({fl})",14)
    y -= 6
    line("CURRENT MEDICATIONS",18,"Helvetica-Bold",11)
    for m in p["meds"]: line(f"  - {m}",14)
    c.showPage(); c.save(); return path

paths = [make_pdf(p) for p in PATIENTS]
print("Generated", len(paths), "synthetic PDFs.")


Generated 3 synthetic PDFs.


In [5]:
import re
from pypdf import PdfReader

def load_pdfs(folder):
    docs = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"): continue
        for page_num, page in enumerate(PdfReader(os.path.join(folder,fname)).pages, start=1):
            text = (page.extract_text() or "").strip()
            if text: docs.append({"source":fname,"page":page_num,"text":text})
    return docs

SECTION_HEADERS = ["CLINICAL NOTE","LABORATORY RESULTS","CURRENT MEDICATIONS"]
def split_sections(text):
    pattern = "(" + "|".join(re.escape(h) for h in SECTION_HEADERS) + ")"
    parts = re.split(pattern, text); sections = []; i = 1
    while i < len(parts):
        name = parts[i]; body = parts[i+1] if i+1 < len(parts) else ""
        sections.append((name, body.strip())); i += 2
    if parts[0].strip(): sections.insert(0, ("HEADER", parts[0].strip()))
    return sections

def extract_mrn(text):
    m = re.search(r"MRN[-:\s]*([0-9]{6})", text); return f"MRN-{m.group(1)}" if m else "UNKNOWN"

def word_chunks(text, max_words=120, overlap=25):
    words = text.split()
    if len(words) <= max_words: return [text]
    out, start = [], 0
    while start < len(words):
        out.append(" ".join(words[start:start+max_words])); start += max_words - overlap
    return out

def build_chunks(docs):
    chunks = []
    for d in docs:
        mrn = extract_mrn(d["text"])
        for sect_name, sect_text in split_sections(d["text"]):
            if not sect_text: continue
            for piece in word_chunks(sect_text):
                chunks.append({"id":len(chunks),"text":piece,"source":d["source"],
                               "page":d["page"],"patient_mrn":mrn,"section":sect_name})
    return chunks

chunks = build_chunks(load_pdfs(DOCS_DIR))
print(f"Built {len(chunks)} chunks.")


Built 12 chunks.


In [6]:
import chromadb
import time # Import time for potential future use or debugging
import requests # Import requests to catch ConnectionError

def embed_texts(texts, task_type, batch_size=10): # Reduced batch_size to 10
    vecs = []
    for i in range(0, len(texts), batch_size):
        retries = 3
        delay = 5 # Initial delay in seconds
        for attempt in range(retries):
            try:
                # Increase the timeout for the embed_content call to 300 seconds (5 minutes)
                resp = genai.embed_content(model=EMBED_MODEL, content=texts[i:i+batch_size], task_type=task_type, request_options={"timeout": 300})
                vecs.extend(resp["embedding"])
                break # If successful, break out of the retry loop
            except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
                print(f"Embedding attempt {attempt + 1}/{retries} failed: {e}")
                if attempt < retries - 1:
                    print(f"Retrying in {delay} seconds...")
                    time.sleep(delay)
                    delay *= 2 # Exponential backoff
                else:
                    raise # Re-raise the exception if all retries fail
        time.sleep(1) # Add a small delay between batches to avoid overwhelming the service
    return vecs

client = chromadb.Client()  # in-memory; use chromadb.PersistentClient(path="...") to persist to disk
try:
    client.delete_collection("patient_records")
except Exception:
    pass
collection = client.create_collection(name="patient_records", metadata={"hnsw:space": "cosine"})

embeddings = embed_texts([c["text"] for c in chunks], task_type="retrieval_document")
collection.add(
    ids=[str(c["id"]) for c in chunks],
    embeddings=embeddings,
    documents=[c["text"] for c in chunks],
    metadatas=[{"patient_mrn":c["patient_mrn"], "section":c["section"],
                "source":c["source"], "page":c["page"]} for c in chunks],
)
print("ChromaDB collection 'patient_records' now holds", collection.count(), "vectors.")

ChromaDB collection 'patient_records' now holds 12 vectors.


In [7]:
def retrieve(query, k=4, patient_mrn=None):
    qv = embed_texts([query], task_type="retrieval_query")[0]
    where = {"patient_mrn": patient_mrn} if patient_mrn else None
    res = collection.query(query_embeddings=[qv], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text":doc, "meta":meta, "similarity":1.0 - dist})
    return hits

for h in retrieve("What is the patient's dialysis access?", k=3):
    m = h["meta"]
    print(f"{h['similarity']:.3f}  {m['patient_mrn']} | {m['section']} | {m['source']} p{m['page']}")


0.747  MRN-100245 | CLINICAL NOTE | MRN_100245.pdf p1
0.722  MRN-101533 | CLINICAL NOTE | MRN_101533.pdf p1
0.717  MRN-100871 | CLINICAL NOTE | MRN_100871.pdf p1


In [8]:
SYSTEM_RULES = """You are a clinical documentation assistant for kidney care.
Answer ONLY using the CONTEXT below. Rules:
1. After every factual claim, cite the source as [filename, p.PAGE].
2. If the answer is not in the context, reply exactly: "I don't have that information in the provided records."
3. Never guess lab values, doses, or diagnoses. Be concise and clinical.
"""

def format_context(hits):
    return "\n\n".join(
        f"[{h['meta']['source']}, p.{h['meta']['page']}] (patient {h['meta']['patient_mrn']}, {h['meta']['section']})\n{h['text']}"
        for h in hits)

def answer(query, k=4, patient_mrn=None):
    hits = retrieve(query, k=k, patient_mrn=patient_mrn)
    prompt = f"{SYSTEM_RULES}\n\nCONTEXT:\n{format_context(hits)}\n\nQUESTION: {query}\n\nANSWER:"
    text = genai.GenerativeModel(CHAT_MODEL).generate_content(prompt).text.strip()
    return text, hits

ans, hits = answer("What dialysis access does John Carter have?", patient_mrn="MRN-100245")
print(ans)


John Carter has a left upper-arm AV fistula for dialysis access [MRN_100245.pdf, p.1].


In [9]:
print('--- Testing the `retrieve` function directly ---')
query = "What is the main kidney condition of Robert T. Nguyen?"
retrieved_hits = retrieve(query, k=2, patient_mrn="MRN-101533")
for i, h in enumerate(retrieved_hits):
    m = h["meta"]
    print(f"\nHit {i+1} (similarity: {h['similarity']:.3f}):")
    print(f"  Patient MRN: {m['patient_mrn']}")
    print(f"  Section: {m['section']}")
    print(f"  Source: {m['source']} p{m['page']}")
    print(f"  Text: {h['text'][:150]}...")

--- Testing the `retrieve` function directly ---

Hit 1 (similarity: 0.751):
  Patient MRN: MRN-101533
  Section: HEADER
  Source: MRN_101533.pdf p1
  Text: RIVER VALLEY NEPHROLOGY ASSOCIATES  -  EXTERNAL CLINIC RECORD
Patient: Robert T. Nguyen    MRN: MRN-101533
DOB: 1949-06-05    Age: 76...

Hit 2 (similarity: 0.664):
  Patient MRN: MRN-101533
  Section: CLINICAL NOTE
  Source: MRN_101533.pdf p1
  Text: Home Peritoneal Dialysis Follow-up. ESRD on automated peritoneal dialysis (APD) nightly.
Etiology: polycystic kidney disease. PD catheter exit site cl...


The output above shows the raw `hits` from the `retrieve` function. These are the relevant chunks of text from the documents that the system finds based on your query. The `answer` function then uses these `hits` as `CONTEXT` for the language model to generate a coherent answer and cite the sources.

In [10]:
import gradio as gr

PATIENT_CHOICES = ["All patients"] + [f"{p['name']} ({p['mrn']})" for p in PATIENTS]
def choice_to_mrn(choice):
    if choice == "All patients": return None
    return choice.split("(")[-1].rstrip(")")

def chat(question, patient_choice):
    if not question.strip():
        return "Please enter a question.", ""
    mrn = choice_to_mrn(patient_choice)
    text, hits = answer(question, k=4, patient_mrn=mrn)
    sources = "\n".join(
        f"- {h['meta']['source']} p{h['meta']['page']} ({h['meta']['section']}, sim={h['similarity']:.2f})"
        for h in hits)
    return text, sources

with gr.Blocks(title="DaVita IKC — Clinical RAG") as demo:
    gr.Markdown("## DaVita IKC — Patient-Records RAG\nAsk questions grounded in the (synthetic) patient charts. "
                "Answers cite their sources and refuse when the info isn't in the records.")
    with gr.Row():
        patient = gr.Dropdown(PATIENT_CHOICES, value="All patients", label="Patient scope")
    question = gr.Textbox(label="Question", placeholder="e.g. What is the patient's dialysis access and Kt/V?")
    btn = gr.Button("Ask", variant="primary")
    answer_box = gr.Markdown(label="Answer")
    sources_box = gr.Textbox(label="Retrieved sources", lines=4)
    btn.click(chat, inputs=[question, patient], outputs=[answer_box, sources_box])
    question.submit(chat, inputs=[question, patient], outputs=[answer_box, sources_box])

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4889930d7bd36bd777.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
